# T8 extractor notebook-check (real LLM)

Runs the shipped `classify` (Haiku) + `extract` (Sonnet) on real fixture emails and prints each `ExtractedFields`. This is the check the mocked unit tests can't give: does the extractor keep `company_raw`/`role_raw` verbatim and produce sane fields on actual email text?

**Live, billable calls.** Probe only — not shipped code.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.environ['ANTHROPIC_API_KEY']

from email_parser.source import FixtureSource
from email_parser.classify import classify
from email_parser.extract_fields import extract

emails = FixtureSource('tests/fixtures/emails').fetch()
print(len(emails), 'fixture emails')
for e in emails:
    print(' -', e.gmail_message_id, '|', e.subject)

5 fixture emails
 - msg-recruiter-001 | Exciting Backend Engineer role at Acme
 - msg-interview-002 | Interview invitation — Data Scientist, Acme
 - msg-rejection-003 | Update on your application to Globex
 - msg-confirm-004 | We received your application
 - msg-other-005 | Your job search newsletter for this week


In [2]:
def probe(email):
    cat = classify(email, api_key=API_KEY)          # real Haiku
    fields = extract(email, cat, api_key=API_KEY)   # real Sonnet
    print('=' * 70)
    print('email  :', email.gmail_message_id, '|', email.subject)
    print('sender :', email.sender)
    print('category (Haiku):', cat.value)
    print('-- ExtractedFields (Sonnet) --')
    print('  company_raw          :', repr(fields.company_raw))
    print('  role_raw             :', repr(fields.role_raw))
    print('  contact_name         :', repr(fields.contact_name))
    print('  action_required      :', fields.action_required)
    print('  action_description   :', repr(fields.action_description))
    print('  extraction_confidence:', fields.extraction_confidence)
    print('  key_dates:')
    for kd in fields.key_dates:
        print('     -', kd.type.value, '| date=', kd.date, '| raw=', repr(kd.raw_text))
    if not fields.key_dates:
        print('     (none)')
    return cat, fields

In [3]:
_ = probe(emails[0])

email  : msg-recruiter-001 | Exciting Backend Engineer role at Acme
sender : Jordan Lee <jordan@acmetalent.example>
category (Haiku): recruiter_outreach
-- ExtractedFields (Sonnet) --
  company_raw          : 'Acme'
  role_raw             : 'Senior Backend Engineer'
  contact_name         : 'Jordan Lee'
  action_required      : True
  action_description   : 'Reply to Jordan Lee to indicate whether you are open to new roles.'
  extraction_confidence: high
  key_dates:
     (none)


In [4]:
_ = probe(emails[1])

email  : msg-interview-002 | Interview invitation — Data Scientist, Acme
sender : Acme Recruiting <recruiting@acme.example>
category (Haiku): interview_invite
-- ExtractedFields (Sonnet) --
  company_raw          : 'Acme'
  role_raw             : 'Data Scientist'
  contact_name         : 'Acme Recruiting'
  action_required      : True
  action_description   : 'Confirm availability for the first-round interview on Tuesday, August 4 at 3pm PT.'
  extraction_confidence: high
  key_dates:
     - interview | date= 2026-08-04 | raw= 'Tuesday, August 4 at 3pm PT'


In [5]:
_ = probe(emails[2])

email  : msg-rejection-003 | Update on your application to Globex
sender : no-reply@globex.example
category (Haiku): rejection
-- ExtractedFields (Sonnet) --
  company_raw          : 'Globex'
  role_raw             : 'ML Engineer'
  contact_name         : None
  action_required      : False
  action_description   : None
  extraction_confidence: high
  key_dates:
     (none)


In [6]:
_ = probe(emails[3])

email  : msg-confirm-004 | We received your application
sender : jobs@initech.example
category (Haiku): application_confirmation
-- ExtractedFields (Sonnet) --
  company_raw          : 'Initech'
  role_raw             : 'Platform Engineer'
  contact_name         : None
  action_required      : False
  action_description   : None
  extraction_confidence: high
  key_dates:
     (none)


In [7]:
_ = probe(emails[4])

email  : msg-other-005 | Your job search newsletter for this week
sender : digest@jobsboard.example
category (Haiku): other
-- ExtractedFields (Sonnet) --
  company_raw          : None
  role_raw             : None
  contact_name         : None
  action_required      : False
  action_description   : None
  extraction_confidence: high
  key_dates:
     (none)


## Eyeball checklist

- `company_raw` / `role_raw` are copied **verbatim** from the email (no normalization, no invented legal suffix).
- Missing fields are `None` / `[]` / `False`, not guessed.
- `interview_invite` carries a `key_dates` entry; `rejection` carries none.
- `extraction_confidence` is advisory — a `low` here would still be written.